# 3D Gaussian Splatting (3DGS) on Google Colab
### 3D Computer Vision Capstone Project

**Objective:** Train a continuous, differentiable 3D Gaussian radiance field on the registered 28 SfM keyframes of the smooth, specular bottle (`VID_20261102_015654418.mp4`).

**Theoretical Motivation:**
- Classical PatchMatch MVS assumes Lambertian (matte) reflectance, which struggles with dynamic light glare on shiny plastic/glass.
- 3D Gaussian Splatting models the scene as millions of anisotropic 3D Gaussians with **Spherical Harmonics (SH)** for view-dependent color rendering.
- Uses the exact camera poses ($R, t$) and initial sparse point cloud from your COLMAP Structure-from-Motion pipeline.

**Expected Runtime on Tesla T4:** $\approx 4-6$ minutes for 7,000 iterations.

### Step 1: Verify Tesla T4 GPU Environment

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

### Step 2: Clone Repository & Install Custom CUDA Submodules
Compiles `diff-gaussian-rasterization` and `simple-knn` on the T4 GPU.

In [ ]:
%cd /content
!git clone --recursive https://github.com/camenduru/gaussian-splatting
!pip install -q plyfile tqdm

# Install CUDA rasterization & KNN submodules
!pip install -q /content/gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q /content/gaussian-splatting/submodules/simple-knn

print("3DGS CUDA submodules successfully compiled and installed!")

### Step 3: Prepare the Dataset Directory
Organizes your 28 registered keyframes and COLMAP camera poses (`sparse/0`) into standard 3DGS format.

In [ ]:
import os, shutil
from pathlib import Path

# Ensure files from colab_mvs_input are present
dataset_dir = Path("/content/3dgs_dataset")
images_dest = dataset_dir / "images"
sparse_dest = dataset_dir / "sparse" / "0"

images_dest.mkdir(parents=True, exist_ok=True)
sparse_dest.mkdir(parents=True, exist_ok=True)

# Copy keyframes to images/
kf_dir = Path("/content/keyframes")
if kf_dir.exists():
    for img in kf_dir.glob("*.jpg"):
        shutil.copy(img, images_dest / img.name)

# Copy sparse reconstruction files
sp_dir = Path("/content/sparse")
if sp_dir.exists():
    for bin_file in sp_dir.glob("*.bin"):
        shutil.copy(bin_file, sparse_dest / bin_file.name)

print(f"Dataset prepared at {dataset_dir}:")
print(f"  Images: {len(list(images_dest.glob('*.jpg')))}")
print(f"  Sparse files: {[f.name for f in sparse_dest.glob('*.bin')]}")

### Step 4: Train 3D Gaussian Splatting (3DGS)
Runs 7,000 iterations of differentiable Gaussian rasterization with adaptive density control.

In [ ]:
%cd /content/gaussian-splatting

# Train 3DGS on the prepared dataset
!python train.py -s /content/3dgs_dataset -m /content/3dgs_output --iterations 7000 --eval

### Step 5: Render Novel View Synthesis & Evaluation Metrics
Renders the trained scene and measures PSNR / SSIM quality.

In [ ]:
%cd /content/gaussian-splatting
!python render.py -m /content/3dgs_output
!python metrics.py -m /content/3dgs_output

### Step 6: Package Trained 3DGS Splat Model for Download
Packages the trained Gaussian Splat `.ply` (compatible with online viewers like SuperSplat and Polycam).

In [ ]:
from IPython.display import FileLink
import zipfile
from pathlib import Path

output_zip = Path("/content/3dgs_results.zip")
model_ply = Path("/content/3dgs_output/point_cloud/iteration_7000/point_cloud.ply")

with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as z:
    if model_ply.exists():
        z.write(model_ply, arcname="point_cloud_3dgs_7000.ply")
        print(f"Added {model_ply.name} ({round(model_ply.stat().st_size / (1024*1024), 2)} MB)")
    
    # Also include renders
    renders_dir = Path("/content/3dgs_output/train/ours_7000/renders")
    if renders_dir.exists():
        for img in renders_dir.glob("*.png"):
            z.write(img, arcname=f"renders/{img.name}")
        print("Added rendered novel views.")

print(f"3DGS Results packaged to {output_zip} ({round(output_zip.stat().st_size / (1024*1024), 2)} MB)")
FileLink('3dgs_results.zip')